<a href="https://colab.research.google.com/github/amramakri/BME450-PROJECT/blob/main/BME_450_PROJECT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#Neural Network Song Recommendation System

#Mount my Google Drive
from google.colab import drive
drive.mount('/content/drive')

#Imports
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics.pairwise import cosine_similarity
from ipywidgets import interact, FloatSlider
import warnings
warnings.filterwarnings("ignore")

#Load Dataset From my Google Drive
file_path = '/content/drive/MyDrive/450/song data.csv'
df = pd.read_csv(file_path)

#Check the columns of the csv
df.columns = df.columns.str.strip()
print("Columns:", df.columns.tolist())

#Remove rows with missing essential data
df = df.dropna(subset=[
    'Song length', 'Mood', 'Time of day', 'Tempo', 'Lyricism',
    'Geographical location', 'Energy of listener', 'Weather', 'Activity level',
    'Song title', 'Genre', 'Artist'
])

#Normalize the 9 features
feature_cols = [
    'Song length', 'Mood', 'Time of day', 'Tempo', 'Lyricism',
    'Geographical location', 'Energy of listener', 'Weather', 'Activity level'
]
scaler = MinMaxScaler()
X = scaler.fit_transform(df[feature_cols])

#Fake the preference scores for training the neural network
y = np.linspace(1, 0, len(X))

#Here it converts to PyTorch tensors
X_tensor = torch.tensor(X, dtype=torch.float32)
y_tensor = torch.tensor(y, dtype=torch.float32).view(-1, 1)

#Neural network model
class RecommenderNet(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 32),
            nn.ReLU(),
            nn.Linear(32, 16),
            nn.ReLU(),
            nn.Linear(16, 8)
        )

    def forward(self, x):
        return self.net(x)

model = RecommenderNet(input_dim=X.shape[1])
loss_fn = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)

#Train the system
print("Training model...")
for epoch in range(100):
    model.train()
    optimizer.zero_grad()
    outputs = model(X_tensor)
    loss = loss_fn(outputs.mean(dim=1, keepdim=True), y_tensor)
    loss.backward()
    optimizer.step()
    if epoch % 10 == 0:
        print(f"Epoch {epoch}: Loss = {loss.item():.4f}")

#Creating interactive GUI for top recommendations from 3 different genres
print("Enter your desired song length from 0 (very short) to 1 (very long)")
print("Enter your current mood from 0 (very sad) to 1 (very happy)")
print("Enter the time of day from 0 (early morning) to 1 (late night)")
print("Enter your desired tempo from 0 (very slow) to 1 (very fast)")
print("Enter your desired lyricism from 0 (instrumenta) to 1 (speech only)")
print("Enter your geographical location from 0 (tropical) to 1 (northern)")
print("Enter your current energy level from 0 (low) to 1 (high)")
print("Enter the current weather from 0 (sunny) to 1 (snowy)")
print("Enter your current activity level from 0 (inactive) to 1 (very active)")

@interact(
    song_length=FloatSlider(min=0, max=1, step=0.01, description='Length'),
    mood=FloatSlider(min=0, max=1, step=0.01, description='Mood'),
    time_of_day=FloatSlider(min=0, max=1, step=0.01, description='Time'),
    tempo=FloatSlider(min=0, max=1, step=0.01, description='Tempo'),
    lyricism=FloatSlider(min=0, max=1, step=0.01, description='Lyrics'),
    geo=FloatSlider(min=0, max=1, step=0.01, description='Geo'),
    energy=FloatSlider(min=0, max=1, step=0.01, description='Energy'),
    weather=FloatSlider(min=0, max=1, step=0.01, description='Weather'),
    activity=FloatSlider(min=0, max=1, step=0.01, description='Activity'),
)
def get_input(song_length, mood, time_of_day, tempo, lyricism, geo, energy, weather, activity):
    user_input = [song_length, mood, time_of_day, tempo, lyricism, geo, energy, weather, activity]

    #Normalizing and create the tensor
    user_input_df = pd.DataFrame([user_input], columns=feature_cols)
    user_input_scaled = scaler.transform(user_input_df)
    user_tensor = torch.tensor(user_input_scaled, dtype=torch.float32)
    user_embedding = model(user_tensor).detach().numpy()

    model.eval()
    song_embeddings = model(X_tensor).detach().numpy()
    similarities = cosine_similarity(user_embedding, song_embeddings)[0]

    df['similarity'] = similarities

    #Group by the genre and get top 1 song per genre
    top_songs_per_genre = df.sort_values(by='similarity', ascending=False).groupby('Genre').head(1)

    #Sort and pick top 3 genres
    top_3 = top_songs_per_genre.sort_values(by='similarity', ascending=False).head(3)

    print("\n🎵 Top Recommended Songs from 3 Genres:\n")
    for i, idx in enumerate(top_3.index, 1):
        song = df.loc[idx]
        print(f"{i}. {song['Song title']} by {song['Artist']} ({song['Genre']})")


Mounted at /content/drive
Columns: ['Genre', 'Song title', 'Artist', 'Song length', 'Mood', 'Time of day', 'Tempo', 'Lyricism', 'Geographical location', 'Energy of listener', 'Weather', 'Activity level']
Training model...
Epoch 0: Loss = 0.3887
Epoch 10: Loss = 0.1088
Epoch 20: Loss = 0.0878
Epoch 30: Loss = 0.0824
Epoch 40: Loss = 0.0757
Epoch 50: Loss = 0.0673
Epoch 60: Loss = 0.0582
Epoch 70: Loss = 0.0482
Epoch 80: Loss = 0.0404
Epoch 90: Loss = 0.0368
Enter your desired song length from 0 (very short) to 1 (very long)
Enter your current mood from 0 (very sad) to 1 (very happy)
Enter the time of day from 0 (early morning) to 1 (late night)
Enter your desired tempo from 0 (very slow) to 1 (very fast)
Enter your desired lyricism from 0 (instrumenta) to 1 (speech only)
Enter your geographical location from 0 (tropical) to 1 (northern)
Enter your current energy level from 0 (low) to 1 (high)
Enter the current weather from 0 (sunny) to 1 (snowy)
Enter your current activity level from 0 

interactive(children=(FloatSlider(value=0.0, description='Length', max=1.0, step=0.01), FloatSlider(value=0.0,…